In [ ]:
# importing the necessary libraries and loading the dataset
import pandas as pd
df = pd.read_csv('D:/JOINIT SOLUTIONS DBA COURSE/UBa25PP133 MSC DATA SCIENCE/STUDY COURSES 2ND SEMESTER/03_RECOMMENDER SYSTEMS/NER WITH NLTK/womens_e-commerce_cloth_reviews.csv')
df.head()

,S/n,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [8]:
# Check the columns, shape and first few rows of the dataframe
print(df.columns.tolist())
print(df.shape)
print(df.head(3).to_string())

['S/n', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
(23486, 11)
   S/n  Clothing ID  Age                    Title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Review Text  Rating  Recommended IND  Positive Feedback Count Division Name Department Name Class Name
0    0          767   33                      NaN                                                                                                                                        

In [12]:
# Check for missing values in the dataframe and shape of the dataframe
print('Shape:', df.shape)
print('Nulls/Missing values:')
print(df.isnull().sum())

Shape: (23486, 11)
Nulls/Missing values:
S/n                           0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64


In [17]:
"""
=============================================================
  Lab: NER + Sentiment Analysis on Product Reviews (NLTK)
       + Recommender System using Extracted Features
=============================================================
 
Dataset : Women's E-Commerce Clothing Reviews (23 486 rows)
Tools   : NLTK (tokenizer, POS tagger, NE Chunker, VADER)
          scikit-learn (TF-IDF, cosine similarity)
=============================================================
"""

"\n=============================================================\n  Lab: NER + Sentiment Analysis on Product Reviews (NLTK)\n       + Recommender System using Extracted Features\n=============================================================\n\nDataset : Women's E-Commerce Clothing Reviews (23 486 rows)\nTools   : NLTK (tokenizer, POS tagger, NE Chunker, VADER)\n          scikit-learn (TF-IDF, cosine similarity)\n=============================================================\n"

In [1]:
# Imports and NLTK data downloads
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
print('All NLTK data downloaded')

All NLTK data downloaded


In [3]:
# Importing necessary libraries for NER and Sentiment Analysis
# ── 0. Imports ────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")
 
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import pos_tag, ne_chunk
from nltk.tree import Tree
 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
print("=" * 60)
print("  NER + Sentiment Analysis + Recommender System with NLTK")
print("=" * 60)

  NER + Sentiment Analysis + Recommender System with NLTK


In [4]:
# ── 1. Load & Clean Data ──────────────────────────────────
print("\n[STEP 1] Loading and cleaning data …")
 
df = pd.read_csv('D:/JOINIT SOLUTIONS DBA COURSE/UBa25PP133 MSC DATA SCIENCE/STUDY COURSES 2ND SEMESTER/03_RECOMMENDER SYSTEMS/NER WITH NLTK/womens_e-commerce_cloth_reviews.csv')
df.head()


[STEP 1] Loading and cleaning data …


,S/n,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [5]:
# Keep only rows with a review text
df = df.dropna(subset=["Review Text"]).reset_index(drop=True)
 
# Fill remaining NaNs
df["Title"] = df["Title"].fillna("")
df["Division Name"] = df["Division Name"].fillna("Unknown")
df["Department Name"] = df["Department Name"].fillna("Unknown")
df["Class Name"] = df["Class Name"].fillna("Unknown")
 
print(f"  ✓ Loaded {len(df):,} reviews after cleaning")
print(f"  Columns: {list(df.columns)}")
print(f"\n  Sample review:\n  '{df['Review Text'].iloc[0][:120]}…'\n")

  ✓ Loaded 22,641 reviews after cleaning
  Columns: ['S/n', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']

  Sample review:
  'Absolutely wonderful - silky and sexy and comfortable…'



In [6]:
# ── 2. Text Pre-processing ────────────────────────────────
print("[STEP 2] Pre-processing text …")
 
STOP_WORDS = set(stopwords.words("english"))
 
def clean_text(text):
    """Lowercase, remove special chars, strip extra spaces."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
def tokenize_and_filter(text):
    """Tokenize and remove stop-words."""
    tokens = word_tokenize(clean_text(text))
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
 
df["cleaned_review"] = df["Review Text"].apply(clean_text)
df["tokens"] = df["Review Text"].apply(tokenize_and_filter)
 
print(f"  ✓ Sample tokens: {df['tokens'].iloc[1][:10]}")

[STEP 2] Pre-processing text …
  ✓ Sample tokens: ['love', 'dress', 'sooo', 'pretty', 'happened', 'find', 'store', 'glad', 'never', 'would']


In [7]:
# ── 3. Sentiment Analysis with VADER ─────────────────────
print("\n[STEP 3] Running VADER Sentiment Analysis …")
 
sia = SentimentIntensityAnalyzer()
 
def get_sentiment(text):
    """Return compound score and sentiment label."""
    score = sia.polarity_scores(text)["compound"]
    if score >= 0.05:
        return score, "Positive"
    elif score <= -0.05:
        return score, "Negative"
    else:
        return score, "Neutral"


[STEP 3] Running VADER Sentiment Analysis …


In [8]:
# Apply VADER to every review
sentiments = df["Review Text"].apply(get_sentiment)
df["sentiment_score"]  = sentiments.apply(lambda x: x[0])
df["sentiment_label"]  = sentiments.apply(lambda x: x[1])
 
# Distribution
counts = df["sentiment_label"].value_counts()
print(f"\n  Sentiment Distribution:")
for label, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"    {label:<10} {count:>6,}  ({pct:.1f}%)  {bar}")
 
# Average sentiment by rating (sanity check)
print(f"\n  Average VADER score by Star Rating:")
avg_by_rating = df.groupby("Rating")["sentiment_score"].mean()
for rating, avg in avg_by_rating.items():
    print(f"    {rating}★ → {avg:+.3f}")


  Sentiment Distribution:
    Positive   20,987  (92.7%)  ██████████████████████████████████████████████
    Negative    1,377  (6.1%)  ███
    Neutral       277  (1.2%)  

  Average VADER score by Star Rating:
    1★ → +0.208
    2★ → +0.400
    3★ → +0.539
    4★ → +0.744
    5★ → +0.853


In [10]:
# ── 4. Named Entity Recognition (NER) ────────────────────
print("\n[STEP 4] Running NER on a sample of reviews …")
 
def extract_named_entities(text):
    """
    Use NLTK's pipeline:
      tokenize → POS tag → NE chunk → extract entity strings
    Returns a list of (entity_text, entity_type) tuples.
    """
    entities = []
    for sentence in sent_tokenize(text):
        tokens = word_tokenize(sentence)
        tagged = pos_tag(tokens)               # POS tagging
        chunked = ne_chunk(tagged)             # NE chunking
        for subtree in chunked:
            if isinstance(subtree, Tree):      # Named entity found
                entity_name = " ".join(w for w, t in subtree.leaves())
                entity_type = subtree.label()  # PERSON, ORG, GPE, etc.
                entities.append((entity_name, entity_type))
    return entities

# Run NER on a 500-review sample (full dataset would be slow)
SAMPLE_SIZE = 500
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).copy()
sample_df["named_entities"] = sample_df["Review Text"].apply(extract_named_entities)
 
# Aggregate all entities found
all_entities = [ent for ents in sample_df["named_entities"] for ent in ents]
entity_series = pd.Series(all_entities)


[STEP 4] Running NER on a sample of reviews …


In [11]:
# Count by type
from collections import Counter
type_counts = Counter(etype for _, etype in all_entities)
print(f"\n  Entity types found in {SAMPLE_SIZE} reviews:")
for etype, cnt in type_counts.most_common():
    print(f"    {etype:<12} → {cnt} occurrences")
 
# Top entities overall
top_entities = Counter(name for name, _ in all_entities).most_common(10)
print(f"\n  Top 10 named entities:")
for ent, cnt in top_entities:
    print(f"    '{ent}' → {cnt}×")
 
# Show a concrete example
example_idx = sample_df[sample_df["named_entities"].apply(len) > 0].index[0]
example_text  = df.loc[example_idx, "Review Text"]
example_ents  = sample_df.loc[example_idx, "named_entities"]
print(f"\n  Example review excerpt:")
print(f"  '{example_text[:200]}…'")
print(f"  Entities found: {example_ents}")


  Entity types found in 500 reviews:
    GPE          → 39 occurrences
    PERSON       → 1 occurrences

  Top 10 named entities:
    'Great' → 6×
    'Beautiful' → 5×
    'Perfect' → 3×
    'Super' → 2×
    'Adorable' → 2×
    'Gorgeous' → 2×
    'Nice' → 2×
    'Cute' → 2×
    'Love' → 2×
    'Fancy' → 1×

  Example review excerpt:
  'Super comfortable and on trend with the embroidery and beading - wore it day after i received to the airport over tee shirt & jeans and got compliments left and right - love it!!…'
  Entities found: [('Super', 'GPE')]


In [12]:
# ── 5. Feature Engineering for Recommender ───────────────
print("\n[STEP 5] Building features for the Recommender …")
 
"""
We combine THREE signals into a rich feature string for each item:
  a) TF-IDF on the review text   → captures what reviewers say
  b) Sentiment score             → positive vs negative signal
  c) Category metadata           → Department, Class, Division
"""
 
def build_feature_string(row):
    """Merge cleaned review + category info into one string."""
    meta = f"{row['Department Name']} {row['Class Name']} {row['Division Name']}"
    # Repeat metadata to give it more weight in TF-IDF
    return f"{row['cleaned_review']} {meta} {meta}"
 
df["feature_string"] = df.apply(build_feature_string, axis=1)
 
# Aggregate to clothing-item level (one row per Clothing ID)
item_df = (
    df.groupby("Clothing ID")
    .agg(
        avg_sentiment  = ("sentiment_score",  "mean"),
        avg_rating     = ("Rating",            "mean"),
        review_count   = ("Review Text",       "count"),
        recommend_rate = ("Recommended IND",   "mean"),
        # Concatenate all reviews for the item
        combined_text  = ("feature_string",    lambda x: " ".join(x)),
        department     = ("Department Name",   "first"),
        class_name     = ("Class Name",        "first"),
    )
    .reset_index()
)
 
print(f"  ✓ {len(item_df):,} unique clothing items found")
print(f"  ✓ Columns: {list(item_df.columns)}")


[STEP 5] Building features for the Recommender …
  ✓ 1,179 unique clothing items found
  ✓ Columns: ['Clothing ID', 'avg_sentiment', 'avg_rating', 'review_count', 'recommend_rate', 'combined_text', 'department', 'class_name']


In [13]:
# ── 6. TF-IDF Vectorization ───────────────────────────────
print("\n[STEP 6] Vectorising item reviews with TF-IDF …")
 
tfidf = TfidfVectorizer(
    max_features=3000,   # keep the top 3 000 terms
    ngram_range=(1, 2),  # unigrams + bigrams
    min_df=2,            # ignore very rare terms
    stop_words="english"
)
 
tfidf_matrix = tfidf.fit_transform(item_df["combined_text"])
print(f"  ✓ TF-IDF matrix shape: {tfidf_matrix.shape}  (items × terms)")
 
 
# ── 7. Content-Based Recommender ─────────────────────────
print("\n[STEP 7] Building Content-Based Recommender …")
 
# Cosine similarity between all items
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"  ✓ Similarity matrix shape: {cosine_sim.shape}")
 
# Map Clothing ID → matrix index
id_to_idx = pd.Series(item_df.index, index=item_df["Clothing ID"])
 
 
def recommend(clothing_id, top_n=5):
    """
    Given a Clothing ID, return the top_n most similar items.
    Similarity = 70% text cosine sim + 30% sentiment alignment.
    """
    if clothing_id not in id_to_idx:
        return f"Item {clothing_id} not found."
 
    idx = id_to_idx[clothing_id]
    # Text similarity scores
    text_scores = cosine_sim[idx]
 
    # Sentiment similarity (closer average sentiment → higher score)
    target_sent  = item_df.loc[idx, "avg_sentiment"]
    sent_sim     = 1 - np.abs(item_df["avg_sentiment"] - target_sent)
 
    # Weighted combined score
    combined = 0.70 * text_scores + 0.30 * sent_sim.values


[STEP 6] Vectorising item reviews with TF-IDF …
  ✓ TF-IDF matrix shape: (1179, 3000)  (items × terms)

[STEP 7] Building Content-Based Recommender …
  ✓ Similarity matrix shape: (1179, 1179)


In [22]:
# ── Define the Recommendation Function ───────────────────
def recommend(query_id, top_n=5):
    """
    Finds and returns top_n similar items for a given clothing ID.
    """
    # 1. Locate the index of the query item in the DataFrame
    query_indices = item_df[item_df["Clothing ID"] == query_id].index
    if query_indices.empty:
        print(f"Clothing ID {query_id} not found in the dataset.")
        return pd.DataFrame()
    
    idx = query_indices[0]

    # 2. Get your similarity scores for this specific item index
    try:
        # !!! CHANGE 'cosine_sim' HERE TO MATCH YOUR VARIABLE NAME !!!
        combined = cosine_sim[idx] 
    except NameError:
        raise NameError("The variable holding your similarity scores wasn't found. Check its name above!")

    # 3. Rank and exclude the query item itself
    sim_scores = list(enumerate(combined))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx][:top_n]
     
    results = []
    for i, score in sim_scores:
        row = item_df.iloc[i]
        results.append({
            "Clothing ID"   : int(row["Clothing ID"]),
            "Department"    : row["department"],
            "Class"         : row["class_name"],
            "Avg Rating"    : round(row["avg_rating"], 2),
            "Avg Sentiment" : round(row["avg_sentiment"], 3),
            "Review Count"  : int(row["review_count"]),
            "Recommend Rate": f"{row['recommend_rate']*100:.0f}%",
            "Similarity"    : round(score, 4),
        })
                
    return pd.DataFrame(results)
 
 
# ── 8. Demo: Run Recommendations ─────────────────────────
print("\n[STEP 8] Demo — Recommending similar items …")
 
demo_ids = [1078, 862, 1095]    # three random clothing IDs
 
for cid in demo_ids:
    info = item_df[item_df["Clothing ID"] == cid]
    if info.empty:
        print(f"\nQuery Item ID {cid} not found in item_df.")
        continue
    info = info.iloc[0]
    print(f"\n{'─'*60}")
    print(f"  Query Item  : Clothing ID {cid}")
    print(f"  Department  : {info['department']}  |  Class: {info['class_name']}")
    print(f"  Avg Rating  : {info['avg_rating']:.2f}★  |  Reviews: {info['review_count']}")
    print(f"  Avg Sentiment: {info['avg_sentiment']:+.3f}")
    print(f"\n  Top 5 Recommendations:")
    
    recs = recommend(cid, top_n=5)
    if not recs.empty:
        print(recs.to_string(index=False))


[STEP 8] Demo — Recommending similar items …

────────────────────────────────────────────────────────────
  Query Item  : Clothing ID 1078
  Department  : Dresses  |  Class: Dresses
  Avg Rating  : 4.19★  |  Reviews: 987
  Avg Sentiment: +0.740

  Top 5 Recommendations:
 Clothing ID Department   Class  Avg Rating  Avg Sentiment  Review Count Recommend Rate  Similarity
        1094    Dresses Dresses        4.19          0.733           735            82%      0.9980
        1077    Dresses Dresses        4.06          0.716           287            79%      0.9973
        1110    Dresses Dresses        4.27          0.782           471            84%      0.9966
        1080    Dresses Dresses        4.28          0.772           280            83%      0.9964
        1104    Dresses Dresses        4.02          0.721           176            76%      0.9958

────────────────────────────────────────────────────────────
  Query Item  : Clothing ID 862
  Department  : Tops  |  Class: K

In [25]:
# ── 9. Sentiment-Aware Ranking ────────────────────────────
print(f"\n{'='*60}")
print("  Bonus: Top 10 Highest-Sentiment Items (≥5 reviews)")
print(f"{'='*60}")
 
top_sentiment = (
    item_df[item_df["review_count"] >= 5]
    .sort_values("avg_sentiment", ascending=False)
    .head(10)[["Clothing ID", "department", "class_name",
               "avg_rating", "avg_sentiment", "review_count", "recommend_rate"]]
)
top_sentiment.columns = ["ID", "Dept", "Class", "Rating", "Sentiment", "Reviews", "Rec%"]
top_sentiment["Rec%"] = (top_sentiment["Rec%"] * 100).round(0).astype(int).astype(str) + "%"
print(top_sentiment.to_string(index=False))
 
 
print(f"\n{'='*60}")
print("  Bonus: Top 10 Most-Recommended Items")
print(f"{'='*60}")
top_rec = (
    item_df[item_df["review_count"] >= 5]
    .sort_values(["recommend_rate", "avg_rating"], ascending=False)
    .head(10)[["Clothing ID", "department", "class_name",
               "avg_rating", "avg_sentiment", "review_count", "recommend_rate"]]
)
top_rec.columns = ["ID", "Dept", "Class", "Rating", "Sentiment", "Reviews", "Rec%"]
top_rec["Rec%"] = (top_rec["Rec%"] * 100).round(0).astype(int).astype(str) + "%"
print(top_rec.to_string(index=False))
 
 
# ── 10. Save artefacts ────────────────────────────────────
print("\n[STEP 9] Saving output files …")
 
df[["Clothing ID", "Rating", "sentiment_score", "sentiment_label",
    "Department Name", "Class Name", "Review Text"]].to_csv(
    "D:/JOINIT SOLUTIONS DBA COURSE/UBa25PP133 MSC DATA SCIENCE/STUDY COURSES 2ND SEMESTER/03_RECOMMENDER SYSTEMS/NER WITH NLTK/womens_e-commerce_cloth_reviews.csv", index=False)
 
item_df.to_csv("D:/JOINIT SOLUTIONS DBA COURSE/UBa25PP133 MSC DATA SCIENCE/STUDY COURSES 2ND SEMESTER/03_RECOMMENDER SYSTEMS/NER WITH NLTK/item_features.csv", index=False)
 
print("  ✓ reviews_with_sentiment.csv   – every review with VADER score + label")
print("  ✓ item_features.csv            – item-level aggregated features")
 
print(f"\n{'='*60}")
print("  Lab complete! Summary of what was done:")
print("""
  1. Loaded & cleaned 23 486 women's clothing reviews
  2. Tokenised & removed stop-words (NLTK)
  3. VADER Sentiment Analysis → score + label per review
  4. NER via NLTK pipeline (tokenise → POS tag → NE chunk)
  5. Aggregated reviews to item (Clothing ID) level
  6. TF-IDF vectorisation (3 000 features, bigrams)
  7. Cosine similarity matrix for content-based filtering
  8. Recommender: 70% text sim + 30% sentiment alignment
  9. Bonus rankings by sentiment & recommendation rate
""")
print("=" * 60)


  Bonus: Top 10 Highest-Sentiment Items (≥5 reviews)
  ID     Dept      Class   Rating  Sentiment  Reviews Rec%
 715 Intimate     Lounge 5.000000   0.955133        6 100%
 893     Tops Fine gauge 4.200000   0.932300        5  80%
 391 Intimate       Swim 3.833333   0.914850        6  67%
1062  Bottoms      Pants 4.555556   0.909833        9  89%
 382 Intimate       Swim 4.750000   0.907900       12  92%
 167  Bottoms     Shorts 5.000000   0.907043        7 100%
 528  Bottoms     Shorts 4.250000   0.904925        8  88%
 720 Intimate     Lounge 4.800000   0.904160        5 100%
 357  Bottoms     Shorts 4.833333   0.902717        6 100%
1015  Bottoms     Skirts 4.333333   0.901392       12  92%

  Bonus: Top 10 Most-Recommended Items
 ID     Dept    Class  Rating  Sentiment  Reviews Rec%
144 Intimate  Legwear     5.0   0.863883        6 100%
167  Bottoms   Shorts     5.0   0.907043        7 100%
196 Intimate    Sleep     5.0   0.786460        5 100%
378 Intimate     Swim     5.0   0.857